# [STARTER] Udaplay Project

## Part 02 - Agent

In this part of the project, you'll use your VectorDB to be part of your Agent as a tool.

You're building UdaPlay, an AI Research Agent for the video game industry. The agent will:
1. Answer questions using internal knowledge (RAG)
2. Search the web when needed
3. Maintain conversation state
4. Return structured outputs
5. Store useful information for future use

### Setup

In [ ]:
# Only needed for Udacity workspace

import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [10]:


import os
from dotenv import load_dotenv
import chromadb
from lib.agents import Agent
from lib.llm import LLM
from lib.messages import UserMessage, SystemMessage, ToolMessage, AIMessage
from lib.tooling import tool



In [11]:

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")
model = LLM()

### Tools

Build at least 3 tools:
- retrieve_game: To search the vector DB
- evaluate_retrieval: To assess the retrieval performance
- game_web_search: If no good, search the web


#### Retrieve Game Tool

In [12]:


# It should use chroma client and collection you created
chroma_client = chromadb.PersistentClient(path="chromadb")
collection = chroma_client.get_collection("udaplay2")

@tool
def retrieve_game(query: str) -> list[dict]:
    """Semantic search: Finds the most relevant games in the vector database.

    Args:
        query: A question about the game industry.

    Returns:
        A list of games containing Platform, Name, YearOfRelease, and Description.
    """
    query = query.strip()
    if not query:
        raise ValueError("query must not be empty")

    results = collection.query(
        query_texts=[query],
        n_results=5,
        include=["documents", "metadatas"],
    )

    documents = results.get("documents", [[]])[0]
    metadatas = results.get("metadatas", [[]])[0]

    result = []
    for document, metadata in zip(documents, metadatas):
        result.append({
            "Platform": metadata.get("Platform"),
            "Name": metadata.get("Name"),
            "YearOfRelease": metadata.get("YearOfRelease"),
            "Description": metadata.get("Description", document),
        })
    return result

#### Evaluate Retrieval Tool

In [13]:

from pydantic import create_model

@tool
def evaluate_retrieval(
    question: str,
    retrieved_docs: list[dict],
) -> dict:
    """Based on the user's question and on the list of retrieved documents,
    analyze whether the documents are useful to respond to that question.

    Args:
        question: Original question from the user.
        retrieved_docs: Documents retrieved from the vector database.

    Returns:
        An EvaluationReport containing whether the documents are useful
        and an explanation of the evaluation.
    """

    prompt = f"""
Your task is to evaluate whether the retrieved documents are sufficient
to answer the user's question.

User question:
{question}

Retrieved documents:
{retrieved_docs}

Evaluate whether the retrieved documents contain enough relevant information
to answer the question accurately.

Return:
1. useful: true if the documents are sufficient and relevant.
2. useful: false if the documents are insufficient, irrelevant, or missing
   important information.
3. description: Give a detailed explanation of your evaluation so that
   another system can decide whether to accept the retrieval results.

Do not answer the user's question.
Only evaluate the quality and usefulness of the retrieved documents.

Return JSON with exactly these fields:
{{
    "useful": true or false,
    "description": "detailed explanation"
}}

The "useful" value must be either true or false.
Do not return markdown or any text outside the JSON object.
"""
    EvaluationReport = create_model(
        "EvaluationReport",
        useful=(bool, ...),
        description=(str, ...),
    )

    result = model.invoke(prompt,response_format=EvaluationReport)
    return EvaluationReport.model_validate_json(result.content)
   

#### Game Web Search Tool

In [14]:
from tavily import TavilyClient

tavily_client = TavilyClient(api_key=TAVILY_API_KEY)


@tool
def game_web_search(question: str) -> list[dict]:
    """Semantic search: Finds most results in the web.

    Args:
        question: A question about game industry.

    Returns:
        A list of web search results.
    """
    question = question.strip()

    if not question:
        raise ValueError("question must not be empty")

    response = tavily_client.search(
        query=question
    )

    return response.get("results", [])

### Agent

In [17]:

from lib.state_machine import EntryPoint, Step, Termination,StateMachine, Resource
from typing import TypedDict
from lib.vector_db import VectorStoreManager
from lib.memory import LongTermMemory,MemoryFragment

class GameAgentState(TypedDict, total=False):
    question: str
    owner: str
    namespace: str
    memories: list
    retrieved_docs: list[dict]
    evaluation: dict
    web_results: list[dict]
    answer: str

   

entry = EntryPoint()

memory_step = Step(
    "retrieve_memory",
    lambda state, resource: {
        "memories": resource.vars["memory"].search(
            query_text=state["question"],
            owner=state.get("owner", "default"),
            namespace=state.get("namespace", "default")
        ).fragments
    }
)

retrieve_step = Step(
    "retrieve_games",
    lambda state: {
        "retrieved_docs": retrieve_game(state["question"])
    }
)

evaluate_step = Step(
    "evaluate_retrieval",
    lambda state: {
        "evaluation": evaluate_retrieval(
            state["question"],
            state["retrieved_docs"]
        )
    }
)

web_search_step = Step(
    "web_search",
    lambda state: {
        "web_results": game_web_search(state["question"])
    }
)

def store_web_memory(state, resource):
    for result in state.get("web_results", []):
        content = result.get("content", "")

        if content:
            memory_fragment = MemoryFragment(
                content=content,
                owner=state.get("owner", "default"),
                namespace=state.get("namespace", "default")
            )

            resource.vars["memory"].register(memory_fragment)

    return {}

def retrieval_decision(state):
    if state["evaluation"].useful:
        return "generate_answer"

    return "web_search"

def generate_answer(state):
    prompt = f"""
    You are a game industry research assistant.

    Answer the user's question using the information below.

    Question:
    {state["question"]}

    Previous relevant memories:
    {state.get("memories", [])}

    Retrieved documents:
    {state.get("retrieved_docs", [])}

    Web search results:
    {state.get("web_results", [])}

    Give an accurate and concise answer.
    Do not invent information.
    """

    response = model.invoke(prompt)
    
    return {
        "answer": response.content
    }

answer_step = Step(
"generate_answer",
generate_answer
)
store_web_memory_step = Step(
    "store_web_memory",
    store_web_memory
)

termination = Termination()



machine = StateMachine(GameAgentState)
db = VectorStoreManager(OPENAI_API_KEY)
memory = LongTermMemory(db) 

machine.add_steps([
    entry,
    memory_step,
    retrieve_step,
    evaluate_step,
    web_search_step,
    store_web_memory_step,
    answer_step,
    termination,
])

machine.connect(entry, memory_step)
machine.connect(memory_step, retrieve_step)
machine.connect(retrieve_step, evaluate_step)
machine.connect(
    evaluate_step,
    [web_search_step, answer_step],
    condition=retrieval_decision
)

machine.connect(web_search_step, store_web_memory_step)
machine.connect(store_web_memory_step, answer_step)
machine.connect(answer_step, termination)

In [20]:

result = machine.run({
    "question": "When Pokémon Gold and Silver was released?"
},resource=Resource(
        vars={
            "memory": memory
        }
    ))

print(result.get_final_state())

result = machine.run({
    "question": "Which one was the first 3D platformer Mario game?"
},resource=Resource(
        vars={
            "memory": memory
        }
    ))

print(result.get_final_state())

result = machine.run({
    "question": "Was Mortal Kombat X realeased for Playstation 5?"
},resource=Resource(
        vars={
            "memory": memory
        }
    ))

print(result.get_final_state())

result = machine.run(
    {
        "question": "What platforms was Mortal Kombat X released on?",
        "owner": "user123",
        "namespace": "default"
    },
    resource=Resource(
        vars={
            "memory": memory
        }
    )
)

print(result.get_final_state())

[StateMachine] Starting: __entry__
[StateMachine] Executing step: retrieve_memory
[StateMachine] Executing step: retrieve_games
[StateMachine] Executing step: evaluate_retrieval
[StateMachine] Executing step: generate_answer
[StateMachine] Terminating: __termination__
{'question': 'When Pokémon Gold and Silver was released?', 'memories': [MemoryFragment(content='Mortal Kombat X set to release April 14th. Mortal Kombat X was announced 9 years ago. Mortal Kombat (2021) released 5 years ago today (US', owner='default', namespace='default', timestamp=1790163671), MemoryFragment(content="|  |\n\n| Gallery |\n\nMortal Kombat X is both the 10th fighting game and the 21st installment in the Mortal Kombat series. It was developed by NetherRealm Studios for PlayStation 4 and Xbox One and by High Voltage Software and later QLOC S.A. for PC. The current-generation console versions were released on April 14, 2015. [...] Mortal Kombat X is the first game in the Mortal Kombat series to come to PlaySt

### (Optional) Advanced

In [ ]:
# TODO: Update your agent with long-term memory
# TODO: Convert the agent to be a state machine, with the tools being pre-defined nodes